In [2]:
import inspect

def example_tool(input_1: str, input_2: int = 1):
    """docstring for example_tool"""
    return

print(f"function name: {example_tool.__name__}")
print(f"function docstring: {example_tool.__doc__}")
print(f"function signature: {inspect.signature(example_tool)}")

function name: example_tool
function docstring: docstring for example_tool
function signature: (input_1: str, input_2: int = 1)


In [ ]:
""" Convert a function to an input schema """
def function_to_input_schema(func) -> dict:
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }
    try:
        signature = inspect.signature(func)                     # A
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {e}"
        )

    parameters = {}
    for param in signature.parameters.values():
        param_type = type_map.get(param.annotation, "string")    # B
        parameters[param.name] = {"type": param_type}            # B

    required = [
        param.name                                               # C
        for param in signature.parameters.values()               # C
        if param.default == inspect._empty                       # C
    ]

    return {
        "type": "object",
        "properties": parameters,
        "required": required,
    }

In [ ]:
"""Format a tool definition for a function."""
def format_tool_definition(name: str, description: str, parameters: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }


def function_to_tool_definition(func) -> dict:
    return format_tool_definition(
        func.__name__,
        func.__doc__ or "",
        function_to_input_schema(func),
    )


search_tool_definition = function_to_tool_definition(search_web)
print(search_tool_definition)